this script cleans the raw data, cleaning includes following steps
1. remove pubs before 2008  and after 2020

2. remove pub if first author information is missing

3. map raw country name to country code and drop countries with code

4. for each author, take their first affiliation as the real affiliations

5. add author_id information the raw author file

**input**
- article_path: paper information derived from vincent database
- author_path: author information derived from vincent database
- cntry_path: meta country data to standard country name
- pub_author_path: derived from script **clean_demographic.ipynb** contains paper_id, seq_no(author order) and author id

In [11]:
import pandas as pd

In [7]:
article_path = '../../data/dropbox/LaborDivision/Data/rawdata/article_labordivision.txt'
author_path = '../../data/dropbox/LaborDivision/Data/rawdata/article_cntry_labordivision.txt'
cntry_path = '../../data/dropbox/LaborDivision/Data/additionaldata/cntry_meta.csv'
pub_author_path='../../data/dropbox/LaborDivision/Data/cleanedraw/pub_authorid.pkl'

In [3]:
article_new='../../data/dropbox/LaborDivision/Data/cleanedraw/pubs.pkl'
author_new_path='../../data/dropbox/LaborDivision/Data/cleanedraw/authors.pkl'

In [4]:
df = pd.read_csv(article_path,sep="|",on_bad_lines='skip')

In [5]:
df.head()

,UID,OST_BK,Annee_Bibliographique,Code_Discipline,Titre
0,WOS:000419403800078,69470021,2018,18,From early stress to 12-month development in v...
1,WOS:000513294200019,69470022,2020,77,Evolution of robust high speed optical-flow-ba...
2,WOS:000362235300024,69470023,2015,41,Activation of the GLP-1 Receptor by Liraglutid...
3,WOS:000326278200002,69470024,2013,58,A 7-Year Case of Furosemide-Induced Immune Thr...
4,WOS:000305602900042,69470046,2012,77,Study of semiconducting nanomaterials under pr...


In [9]:
df2 = pd.read_csv(author_path, sep="|", on_bad_lines='skip')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb7 in position 191976: invalid start byte

In [15]:
path = '../../data/dropbox/LaborDivision/Data/rawdata/article_author_info.txt'
file_path = path  # Replace 'your_file.txt' with the actual path to your file

try:
    with open(file_path, 'r') as file:
        for i in range(10):
            line = file.readline()
            if not line:
                break  # Break if there are fewer than 10 lines in the file
            print(line.strip())  # Print each line, removing any leading or trailing whitespace
except FileNotFoundError:
    print(f"The file '{file_path}' does not exist.")
except Exception as e:
    print(f"An error occurred: {e}")

UID|Seq_No|first_name|cluster_id|gender|YFP|npapa
WOS:000385622600056|6|Toshihide|27466162|F|1993|75
WOS:000316803100002|2498|R.|58414215|M|1987|1537
WOS:000616717200001|18|Tom P.|46896531|M|1985|208
WOS:000314407400015|1261|G. D.|1720233|U|1995|288
WOS:000458210800001|5|B. J.|84345908|F|2015|2
WOS:000331957900010|4|Maria|72668954|F|1996|300
WOS:000571818000004|7|Anna|84048556|F|2005|41
WOS:000328588700007|3|Robert D.|7494136|M|2001|17
WOS:000347046200030|1834|L.|62260081|M|2006|1159


In [22]:
author_df=pd.read_csv(author_path,sep="\t",
                   header=0,
                   names=['id','full','first','last','seqno','addrno','cntry'],
                   error_bad_lines=False, warn_bad_lines=False)
author_df.head()

,id,full,first,last,seqno,addrno,cntry
0,79079869,"Tudor, M. Scarlett",M. Scarlett,Tudor,2.0,3.0,USA
1,30989949,"Tunney, J. J.",J. J.,Tunney,4.0,1.0,Canada
2,30743826,"Kim, Keon-Hee",Keon-Hee,Kim,2.0,2.0,South Korea
3,55406282,"Biron, David",David,Biron,6.0,3.0,USA
4,74854337,"Fortunato, Santo",Santo,Fortunato,2.0,1.0,Italy


In [4]:
pub_df=pd.read_csv(article_path,sep="\t",header = 0,
                   names = ['id','year','dis'],
                   dtype = {'id':'str'},
                   error_bad_lines=False, warn_bad_lines=False)

#### confine analysis to 2008-2020 and remove author information

In [6]:
pub_df=pub_df[pub_df.year.between(2008,2020)]
pub_df=pub_df.dropna()

#### clean names of countries based on table 'cntry_meta' which is manuelly cleaned, authors without country information and countries without country code will be removed from analysis

In [24]:
cntry_df=pd.read_csv(cntry_path)
cntry_df=cntry_df.dropna()
cntrydict = dict(zip(cntry_df.cntry,cntry_df['Code']))
author_df['Code']=author_df['cntry'].map(cntrydict)
author_df=author_df.dropna(subset=['id','full','last','seqno','addrno','cntry','Code'])

#### remove papers without first authors

In [30]:
withfirst=author_df[author_df.seqno==1].id.unique().tolist()
author_df=author_df[author_df.id.isin(withfirst)]

#### define data type of the columns

In [34]:
author_df = author_df.astype({'id':'str','seqno':'int64','addrno':'int64'})
pub_df = pub_df.astype({'year':'int64','dis':'int64'})

#### merge pub information with author information

In [38]:
author_df=author_df.merge(pub_df,on='id')
author_df=author_df[['id','year','dis','full','first','last','seqno','addrno','cntry','Code']]
author_df = author_df[author_df.year.between(2008,2020)]

#### calculate the proportion of publications have author information of each year

In [41]:
filter_df=author_df[['id','year']].drop_duplicates().groupby(['year']).size().reset_index(name='filter')
raw_df=pub_df.groupby(['year']).size().reset_index(name='raw')
meta = filter_df.merge(raw_df,on='year')
meta['p']=meta['filter']/meta['raw']

In [45]:
meta.head(20)

,year,filter,raw,p
0,2008,976993,1157791,0.843842
1,2009,1029298,1208950,0.851398
2,2010,1078290,1250563,0.862244
3,2011,1161052,1330008,0.872966
4,2012,1228905,1398968,0.878437
5,2013,1326422,1475908,0.898716
6,2014,1380998,1518853,0.909237
7,2015,1465429,1576500,0.929546
8,2016,1607154,1636398,0.982129
9,2017,1668342,1696010,0.983686


In [46]:
pub_df = pub_df[pub_df['id'].isin(author_df['id'].unique())]

#### if author is affiliated with multiple institutions, then take the first and remove rest from analysis

In [47]:
author_df = author_df.sort_values(by=['id','seqno','addrno']
                                 ).drop_duplicates(subset=['id','seqno'],keep='first')

#### match authors id to the original author file. base on paper id and sequence number.

In [4]:
pub_authorid=pd.read_pickle(pub_author_path)
pub_authorid=pub_authorid.astype({'id':'str','cluster_id':'str'})
pub_authorid.head()

,id,seqno,cluster_id
0,75871588,2,689940
1,53614014,3,44823147
2,78030267,1,27676457
3,54383423,2,59751143
4,32237142,3,59894652


In [10]:
author_df=author_df.merge(pub_authorid,on=['id','seqno'],how='left')

In [19]:
author_df.to_pickle(author_new_path)

In [1]:
# drop address number/country column from the dataframe

In [48]:
pub_df.to_pickle(article_new)

In [50]:
len(pub_df['id'].unique())

18684905

In [52]:
len(author_df['id'].unique())

18684905

In [11]:
len(pub_df['id'].unique())

20085758

In [10]:
pub_df

,id,year,dis
0,38579295,2012,1
1,38579296,2012,120
2,38579297,2012,84
3,38579302,2012,55
4,38579305,2012,43
...,...,...,...
35164621,84634936,2019,59
35164624,84634941,2010,38
35164625,84634942,2019,1
35164630,84634947,2012,78
